In [1]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

from tsp_backend import polar_to_cartesian, plot_tsp_route, simmulated_annealing_tsp, constant_traffic_factor_func, nearest_neighbor_tsp, calculate_route_cost, make_distance_matrix, check_if_in_polygon

from itertools import combinations, permutations


In [2]:
def traffic_factor_no_traffic(cost : float) -> float:
    return constant_traffic_factor_func(1.0, 0.0, cost)

def traffic_factor_with_traffic(cost : float) -> float:
    return constant_traffic_factor_func(10.0, 4.0, cost)

def tsp_length(nodes_array: np.ndarray) -> float:
    """
    calculate the length of the tsp traveling distance
    
    :param nodes_array: (sorted) array of coordinates of the N nodes
    :type nodes_array: np.ndarray (N, d), N: number of nodes, d: dimension
    :return: total traveling distance
    :rtype: float
    """
    # calculate the pairwise differences between two consecutive nodes, keep in mind to return to start
    differences = nodes_array - np.roll(nodes_array, shift=-1, axis=0)
    # calculate individual distances and sum up
    distances = np.sqrt(np.sum(np.square(differences), axis=1))
    return np.sum(distances)

# Circle

In [3]:
N = 8
R = 1
phi = np.array([2 * np.pi * i / N + np.pi / 2 for i in range(0,N)])

nodes_circle_N_8 = polar_to_cartesian(R, phi)

In [4]:
%%timeit -r 20
# brute force TSP solution
# always start and end at node 0
# get all permutations of the nodes excluding the first one
node_indices = np.arange(1, N)
perms = permutations(node_indices)
tours = nodes_circle_N_8[np.array([[0] + list(p) + [0] for p in perms])]
lengths = np.array([tsp_length(tour) for tour in tours])

min_index = np.argmin(lengths)
min_length = lengths[min_index]

35.6 ms ± 309 μs per loop (mean ± std. dev. of 20 runs, 10 loops each)


In [18]:
%%timeit

optimal_circle_N_8, optimal_cost_circle_N_8 = simmulated_annealing_tsp(
    nodes_circle_N_8, 
    traffic_factor=traffic_factor_no_traffic, 
    start_point_index=0, 
    rejection_threshold=[20,20,20,20],
    max_iter_per_temperature=300,
    plot_cost=False, 
    plot_cost_out_path='plots/convergence_circle_N_8.pdf')

Temperature: 0.1, Current Cost: 10.66780491400445
Current route: [0 7 1 6 4 5 2 3]
num consec rejections: 0, num iter: 0
Reached maximum number of iterations for temperature 0.1. :(
Temperature: 0.05, Current Cost: 6.122934917841436
Current route: [0 1 2 3 4 5 6 7]
num consec rejections: 7, num iter: 301
Temperature: 0.01, Current Cost: 6.122934917841436
Current route: [0 7 6 5 4 3 2 1]
num consec rejections: 23, num iter: 272
Reached maximum number of iterations for temperature 0.01. :(
Temperature: 0.001, Current Cost: 6.122934917841436
Current route: [0 7 6 5 4 3 2 1]
num consec rejections: 6, num iter: 304
Temperature: 0.1, Current Cost: 12.551284746917185
Current route: [0 1 6 2 4 7 5 3]
num consec rejections: 0, num iter: 0
Temperature: 0.05, Current Cost: 6.122934917841436
Current route: [0 1 2 3 4 5 6 7]
num consec rejections: 21, num iter: 91
Temperature: 0.01, Current Cost: 6.122934917841436
Current route: [0 7 6 5 4 3 2 1]
num consec rejections: 24, num iter: 56
Reached maxi

In [6]:
%%timeit -r 20

nearest_neighbor_circle_N_8 = nearest_neighbor_tsp(nodes_circle_N_8, start_point_index=0)

187 μs ± 7.76 μs per loop (mean ± std. dev. of 20 runs, 10,000 loops each)


# Large Circle


In [7]:
N = 100
R = 1
phi = np.array([2 * np.pi * i / N + np.pi / 2 for i in range(0,N)])

nodes_circle_N_100 = polar_to_cartesian(R, phi)

In [8]:
%%timeit -r 20

optimal_circle_N_100, optimal_cost_circle_N_100 = simmulated_annealing_tsp(
    nodes_circle_N_100, 
    traffic_factor=traffic_factor_no_traffic, 
    start_point_index=0, 
    plot_cost=False, 
    plot_cost_out_path='plots/convergence_circle_N_100.pdf')

Temperature: 0.1, Current Cost: 127.7711372339016
Current route: [ 0 91 62 41  5 18 12 30  8 36  7 73 48 98  2 97 82 57 81 44 77 88 74 11
 66 99 87 51 49 54 16  1 69 17 85 21 58 53 70 22 32 60 50  6 26 40 33 42
 80 27 45 90 35 96 68 93 19 71 28 14 83 64 92 34 63 29 84  9 25 65 78 86
 37 20  3 43 61 56 59 67 15 13 10 47 52 39 94 23  4 31 46 75 95 76 24 89
 79 72 55 38]
num consec rejections: 0, num iter: 0
Temperature: 0.05, Current Cost: 41.1260298154719
Current route: [ 0 99  2  4  9 27 19 32 33 31 34 20 22 42 85 37 46 56 65 58 57 55 68 67
 69 74 73 47 48 52 60 61 63 62 70 64 66 51 39 40 43 50 12  3 14 23 18 25
 24 38 35 36 28 29 30 41 53 44 49 59 54 45 26 21 16 17 15 13 93 92 91 89
 83 86 82 88 95 94 96 97 75 78 79 77 71 72 76 80 87 90 84 81  5  6  8  7
 11 10 98  1]
num consec rejections: 32, num iter: 1067
Temperature: 0.01, Current Cost: 11.795638650450831
Current route: [ 0 94 95 96 98 97 93 92 90 91 89 84 85 87 88 86 82 81 80 83 79 78 76 75
 77 74 73 71 72 69 70 67 68 66 62 64 6

In [9]:
%%timeit -r 20

nearest_neighbor_circle_N_100 = nearest_neighbor_tsp(nodes_circle_N_100, start_point_index=0)

13 ms ± 139 μs per loop (mean ± std. dev. of 20 runs, 100 loops each)


# Two seperated squares of equal lattice nodes

In [10]:
N = 10

x_coordinates_1 = np.linspace(-1.1, -0.1, N)
y_coordinates_1 = np.linspace(0.1 , 1.1, N)

X, Y = np.meshgrid(x_coordinates_1, y_coordinates_1)
coordinates_1 = np.column_stack([X.flatten(), Y.flatten()])

x_coordinates_2 = np.linspace(0.1, 1.1, N)
y_coordinates_2 = np.linspace(-1.1 , -0.1, N)

X, Y = np.meshgrid(x_coordinates_2, y_coordinates_2)
coordinates_2 = np.column_stack([X.flatten(), Y.flatten()])

nodes_two_squares = np.append(coordinates_1, coordinates_2, axis=0)

In [11]:
%%timeit -r 20

optimal_two_square_N_10, optimal_two_square_cost_N_10 = simmulated_annealing_tsp(nodes_two_squares, 
                                                                                  traffic_factor=traffic_factor_no_traffic, 
                                                                                  start_point_index=0, 
                                                                                  max_iter_per_temperature=1e5,
                                                                                  plot_cost=False, 
                                                                                  plot_cost_out_path='plots/convergence_two_squares_N_10.pdf')

Temperature: 0.1, Current Cost: 236.40504756763568
Current route: [  0 198 148  54 154 156 118  15 160  16   2  66  49  61  64  69  98 128
 137  95  51   8 185 123  52 104 142 112 120  50 106 188 182 101  24 158
 141 155 122  44  70 157 103  46 134  33 167 130  75 119 192  97 170 191
 189 180  48  32 152  60   9  12  21  37  82 173 199 166  77 126  30  79
 146  47  94 102 168 114 187  29  26  43 186 127 174  58 111 194 136 109
 163  40 143 179  90 124  13 132 107 169  81  38  27 178  73 133 100 161
 135  71  36 151  42  68 138  65  63  86 190  28 193 116   6 149 159  96
  59   7  53 144 105  62  22 181  88  85 145  74 197  25   4  34 165   1
  80 184  72  20  41 147  93 172 195 110  10 164 108  55  17  92 183  87
  14  76  11 129   5  91 196 176  99   3  83  84  39  35  45  67 175 162
 117  78  18 153 177  56  19 140  89 171 121 131 139  31 113  23  57 115
 150 125]
num consec rejections: 0, num iter: 0
Temperature: 0.05, Current Cost: 83.58034026014452
Current route: [  0  41 191 160 

In [12]:
%%timeit -r 20

nearest_neighbor_two_square_N_10 = nearest_neighbor_tsp(nodes_two_squares, start_point_index=0)

51.9 ms ± 290 μs per loop (mean ± std. dev. of 20 runs, 10 loops each)


# Random points in a unit square

In [13]:
N = 100
x = np.random.rand(N)
y = np.random.rand(N)
nodes_random_N_100 = np.column_stack([x, y])

In [14]:
%%timeit -r 20

optimal_random_square_N_100, optimal_random_square_cost_N_100 = simmulated_annealing_tsp(nodes_random_N_100, 
                                                                                  traffic_factor=traffic_factor_no_traffic, 
                                                                                  start_point_index=0, 
                                                                                  max_iter_per_temperature=1e5,
                                                                                  plot_cost=False, 
                                                                                  plot_cost_out_path='plots/convergence_random_squares_N_100.pdf')

Temperature: 0.1, Current Cost: 52.380186940157486
Current route: [ 0  1 60 90 97 70 38 84 40 34 25 24 57 93 12 26 21 20 52 71 92 91 66 32
 62 73 37 17 10 80 69 31  2 65 14 42 74 58 99 48 46 98  7 15 36 61  8 35
 59 88 19 27 22 64 11 76 33 87 75 63 96  3 86 94  4 44 50 28  9 83 53 81
 89 79 49 13 67 82 39 78 18 47 85  5 77 16 45 55 54 29 51 23  6 43 56 41
 72 68 95 30]
num consec rejections: 0, num iter: 0
Temperature: 0.05, Current Cost: 19.259661158556884
Current route: [ 0 69 41 14 99  7 82 91  4 60 44 32 59 29 26 55 83 25 79 64 16 52 43  1
 34  2 45 72 66 86 96 50 46 13  9 87 89 36 18 27 78 88 77 73 63 95 12 74
 48 75 67 53 58 94 98 97 71 22 80 38 11 35 90 47 28 51  8 92 85 31  6 42
 76 65 70 10 56 20 57 40 19 93  5 23 24 30 62 39 81 33  3 54 61 49 17 21
 15 84 68 37]
num consec rejections: 34, num iter: 5766
Reached maximum number of iterations for temperature 0.05. :(
Temperature: 0.01, Current Cost: 10.471865529301912
Current route: [ 0 37 95  2 51 79 84 92 85 64 28 47  8 68 16 

In [15]:
%%timeit -r 20

nearest_neighbor_random_square_N_100 = nearest_neighbor_tsp(nodes_random_N_100, start_point_index=0)

13 ms ± 120 μs per loop (mean ± std. dev. of 20 runs, 100 loops each)


In [16]:
traffic_polygon = np.array([[0.2, 0.8], [0.8, 0.8], [0.8, 0.2], [0.2, 0.2]])

In [17]:
%%timeit -r 20

optimal_random_square_N_100, optimal_random_square_cost_N_100 = simmulated_annealing_tsp(nodes_random_N_100, 
                                                                                  traffic_factor=traffic_factor_with_traffic, 
                                                                                  traffic_start_time=4,
                                                                                  traffic_polygon=traffic_polygon,
                                                                                  start_point_index=0, 
                                                                                  max_iter_per_temperature=1e5,
                                                                                  plot_cost=False, 
                                                                                  plot_cost_out_path='plots/convergence_random_squares_N_100_with_traffic.pdf')

Temperature: 0.1, Current Cost: 86.52454142499876
Current route: [ 0 42 44 84 46 90 14 35 45 48 25 83 64  9 87 77 82 36  3 85 76 60  8 17
 47 18  6 29 57 65 41 98 40 81 20 24 68 30 13 22 70 74 58  2 23 10 16 62
  4 88 15 59 89 93 37 28 49 19 75 53 31 95 39 43 73 97 26 99 86 94 52 51
 50 21 96 66 91 27 80 38 34 92 69 61 72 12 63 56 55  5 71 33 54 11 79  1
 32  7 67 78]
num consec rejections: 0, num iter: 0
Temperature: 0.05, Current Cost: 26.382291773172945
Current route: [ 0 95 86 35 56 45 49 44 32 99 34 47 17 54 90 87 62 38 21  6 73 51 16 63
 41  7 43  1 91 82 88 23 40 93 60 24  4 46 81 36 74 69 31 65 12 96 94 53
 67 48 71 50 57 39 11 18 80 75 13  9 30 19 61 26 29 72 55 59 25 83  2  8
 37 92 64 66 89 78  3 33  5 27 58 10 76 22 98 20 42 97 70 28 68 77 15 14
 85 84 79 52]
num consec rejections: 35, num iter: 1697
Temperature: 0.01, Current Cost: 12.781965273766687
Current route: [ 0 47 95 86 12 49 63 73 44 66 91 82 32 17 52 77 78 90 35 27 87 40 13  9
  5 46  3 24 60 54 61  4 88 43 72 45